# Start

# 🚀 Phase 1 : Modèles de Filtrage Collaboratif (Collaborative Filtering)

## 🎯 Objectif du Notebook
L'objectif de cette étape est de construire et d'évaluer notre premier moteur de recommandation basé sur le **Filtrage Collaboratif**. 
Ce type d'algorithme ne regarde pas le contenu des articles, mais se base uniquement sur le comportement des lecteurs : *"Les utilisateurs qui ont lu cet article ont également lu ceux-ci"*.

Puisque nous travaillons avec des données implicites (Implicit Feedback = de simples clics, pas d'étoiles sur 5) et que notre application finale doit tourner sur une architecture Cloud Serverless (**Azure Functions**), l'enjeu majeur sera la **Scalabilité**.

## 🧠 Algorithmes Mis à l'Épreuve
Afin de justifier nos choix techniques d'ingénierie, nous allons confronter trois méthodes algorithmiques sur nos 3 millions d'interactions :

1. **KNN Users (Memory-Based) via `scikit-surprise`**
   * *Approche :* Calcul des plus proches voisins (similarité Cosinus).
   * *Risque identifié :* Explosion de la RAM due à la création de la gigantesque matrice Utilisateur-Utilisateur.
2. **SVD (Model-Based) via `scikit-surprise`**
   * *Approche :* Factorisation de matrice classique.
   * *Objectif :* Prouver que la factorisation résout le problème de RAM du KNN.
3. **ALS - Alternating Least Squares (Model-Based) via `implicit`**
   * *Approche :* Factorisation de matrices creuses (Sparse Matrices), ultra-optimisée en C++ et conçue spécifiquement pour les données implicites.
   * *Objectif :* Obtenir la meilleure Latence pour la mise en production.

## 📏 Métriques d'Évaluation (Critères d'acceptation)
Pour désigner le modèle gagnant de cette phase, nous appliquerons un banc d'essai (Benchmark) basé sur 3 métriques :
* **Pertinence (Métier) :** Le **Hit Ratio @ 5** calculé via une approche chronologique *Leave-One-Out* (On cache le dernier clic de l'utilisateur et le modèle doit le retrouver dans son Top 5).
* **Vitesse (Cloud) :** La **Latence d'inférence**, qui doit être inférieure à 200 ms pour l'API web.
* **Coût (Cloud) :** L'**Empreinte Mémoire (RAM)** des matrices exportées, qui doit tenir dans les petites instances Azure.


In [1]:
# Installation des librairies indispensables
!pip install implicit scikit-surprise pyarrow

import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
from surprise import Dataset, Reader, SVD, KNNBasic
import time
import os

print("✅ Librairies installées avec succès !")

from google.colab import drive
drive.mount('/content/drive')

# --- MODIFIE CE CHEMIN SELON TON GOOGLE DRIVE ---
# Par exemple : "/content/drive/MyDrive/P10_Data/clicks_full.parquet"
CLICKS_PATH = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/clicks_full.parquet"

print("📂 Chargement du dataset complet...")
clicks_df = pd.read_parquet(CLICKS_PATH)
print(f"✅ Données chargées : {len(clicks_df)} interactions")


✅ Librairies installées avec succès !
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Chargement du dataset complet...
✅ Données chargées : 2988181 interactions


In [2]:
print("🛠️ Préparation du Train/Test Split (Dernier clic = Cible)...")

# On ne garde que les utilisateurs ayant au moins 2 clics
user_counts = clicks_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index

df_valid = clicks_df[clicks_df['user_id'].isin(valid_users)].copy()
df_valid = df_valid.sort_values(['user_id', 'click_timestamp'])

# Le Test Set = le tout dernier clic chronologique de CHAQUE utilisateur
test_set = df_valid.groupby('user_id').tail(1)

# Le Train Set = tout le reste de l'historique
train_set = df_valid.drop(test_set.index)

print(f"✅ Matrice d'entraînement (Train) : {len(train_set)} clics")
print(f"✅ Cibles à deviner (Test) : {len(test_set)} utilisateurs")


🛠️ Préparation du Train/Test Split (Dernier clic = Cible)...
✅ Matrice d'entraînement (Train) : 2665284 clics
✅ Cibles à deviner (Test) : 322897 utilisateurs


# SVD

SVD a besoin d'avoir des notes explicites qui ne sont pas présentes dans notre cas, donc créons des notes artificielles

In [3]:
from surprise import Dataset, Reader, SVD, KNNBasic

# -------------------------------------------------------------------
# FORMULE DE RATING : Basée sur le nombre de sessions uniques
# -------------------------------------------------------------------

# 1. On compte le nombre de 'session_id' uniques pour chaque paire (Utilisateur, Article)
interactions = train_set.groupby(['user_id', 'click_article_id'])['session_id'].nunique().reset_index(name='sessions_count')

# 2. On transforme ce nombre de sessions en note sur 5 étoiles (Plafond à 5)
# Ex: Lu dans 1 session = 1 étoile. Lu dans 3 sessions distinctes = 3 étoiles.
interactions['rating'] = interactions['sessions_count'].apply(lambda x: min(float(x), 5.0))

print("📊 Distribution des nouvelles notes (Basées sur les sessions) :")
print(interactions['rating'].value_counts())

# 3. Création du Dataset pour la librairie Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(interactions[['user_id', 'click_article_id', 'rating']], reader)
trainset_surprise = data.build_full_trainset()



📊 Distribution des nouvelles notes (Basées sur les sessions) :
rating
1.0    2597053
2.0      30015
3.0       1816
4.0        337
5.0        197
Name: count, dtype: int64


In [4]:
# --- Modèle 1 : SVD ---
print("🧠 Entraînement de SVD...")
t0 = time.time()
model_svd = SVD(n_factors=50, random_state=42)
model_svd.fit(trainset_surprise)
print(f"✅ SVD entraîné en {time.time() - t0:.2f} secondes")


🧠 Entraînement de SVD...
✅ SVD entraîné en 21.33 secondes


# KNN

In [ ]:
# --- Modèle 2 : KNN Users ---
print("\n🧠 Entraînement de KNN...")
print("⚠️ ATTENTION : Crash mémoire quasi certain sur Colab (Matrice de Similarité trop énorme).")
try:
    sim_options = {'name': 'cosine', 'user_based': True}
    model_knn = KNNBasic(sim_options=sim_options, random_state=42)
    model_knn.fit(trainset_surprise)
    print("✅ KNN entraîné !")
except MemoryError as e:
    print(f"💥 DÉMONSTRATION RÉUSSIE : CRASH MÉMOIRE. Le KNN Memory-Based n'est pas scalable !")
except Exception as e:
    print(f"💥 ERREUR : {e}")


: 

: 

# ALS

In [5]:
import scipy.sparse as sparse
import implicit

# 1. ALS a besoin que les index aillent de 0 à N de manière continue
# C'est la fameuse notion de "Mapping" que l'on a documentée !
users = interactions['user_id'].astype("category")
items = interactions['click_article_id'].astype("category")

user_codes = users.cat.codes
item_codes = items.cat.codes
clicks = interactions['sessions_count'].astype(float)

# 2. Création de la matrice Item-User puis User-Item
sparse_item_user = sparse.csr_matrix((clicks, (item_codes, user_codes)))
sparse_user_item = sparse_item_user.T.tocsr()

# 3. Entraînement de l'algorithme Alternating Least Squares
print("\n🧠 Entraînement d'ALS...")
t0 = time.time()
model_als = implicit.als.AlternatingLeastSquares(factors=50, iterations=15, regularization=0.1, random_state=42)
model_als.fit(sparse_user_item)
print(f"✅ ALS entraîné en {time.time() - t0:.2f} secondes")



🧠 Entraînement d'ALS...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

✅ ALS entraîné en 53.66 secondes


In [6]:
# ==========================================
# 🏆 MATCH FINAL : SVD (Surprise) VS ALS (Implicit) 🏆
# ==========================================
import time
import sys
import numpy as np

print("📊 Lancement de l'évaluation sur 10000 utilisateurs du Test Set...\n")

# 1. On isole 10000 utilisateurs pour que le SVD ne mette pas 1 semaine à calculer
test_sample = test_set.sample(10000, random_state=42)
test_targets = dict(zip(test_sample['user_id'], test_sample['click_article_id']))
test_users = list(test_targets.keys())

# Variables de suivi
svd_hits = 0
als_hits = 0
svd_latency_total = 0
als_latency_total = 0

# On liste tous les "Inner IDs" connus par Surprise pour faire les prédictions
all_items_inner = [trainset_surprise.to_inner_iid(i) for i in items.cat.categories if i in trainset_surprise._raw2inner_id_items]

for idx, user in enumerate(test_users):
    target_item = test_targets[user]

    # ----------------------------------------
    # EVALUATION SVD (Surprise)
    # ----------------------------------------
    try:
        user_inner = trainset_surprise.to_inner_uid(user)
        t0 = time.time()
        # SVD ne peut pas recommander le Top 5 nativement. 
        # On doit faire une prédiction pour chaque article du catalogue.
        predictions = [model_svd.predict(user_inner, i).est for i in all_items_inner]
        
        # On trie pour garder les 5 meilleurs
        top_5_idx = np.argsort(predictions)[-5:][::-1]
        top_5_items_svd = [trainset_surprise.to_raw_iid(all_items_inner[i]) for i in top_5_idx]
        svd_latency_total += (time.time() - t0)
    
        if target_item in top_5_items_svd:
            svd_hits += 1
    except ValueError:
        pass # Nouvel utilisateur inconnu pour SVD (Cold Start)
        
    # ----------------------------------------
    # EVALUATION ALS (Implicit)
    # ----------------------------------------
    try:
        user_code = users.cat.categories.get_loc(user)
        t0 = time.time()
        
        # ALS possède une fonction C++ native ultra rapide pour le Top N
        als_recs, _ = model_als.recommend(user_code, sparse_user_item[user_code], N=5)
        
        # On retraduit les index internes de l'ALS en vrais IDs d'articles
        top_5_items_als = items.cat.categories[als_recs].tolist()
        als_latency_total += (time.time() - t0)
        
        if target_item in top_5_items_als:
            als_hits += 1
    except KeyError:
        pass # Nouvel utilisateur (Cold Start)

# --- AFFICHAGE DES RÉSULTATS ---
print("="*50)
print("🏁 RÉSULTATS DU BENCHMARK (10000 utilisateurs)")
print("="*50)

print("\n🎯 Pertinence Métier (Hit Ratio @ 5) :")
print(f" - SVD : {(svd_hits / 10000) * 100:.2f} %")
print(f" - ALS : {(als_hits / 10000) * 100:.2f} %")

print("\n⚡ Latence d'inférence (Temps de réponse / utilisateur) :")
print(f" - SVD : {(svd_latency_total / 10000) * 10000:.2f} ms")
print(f" - ALS : {(als_latency_total / 10000) * 10000:.2f} ms")

print("\n💾 Empreinte Mémoire (RAM estimée pour le déploiement) :")
# Taille des matrices générées par les modèles
size_svd = sys.getsizeof(model_svd.pu) + sys.getsizeof(model_svd.qi)
size_als = sys.getsizeof(model_als.user_factors) + sys.getsizeof(model_als.item_factors)
print(f" - SVD : {size_svd / (1024*1024):.2f} Mo")
print(f" - ALS : {size_als / (1024*1024):.2f} Mo")

📊 Lancement de l'évaluation sur 10000 utilisateurs du Test Set...

🏁 RÉSULTATS DU BENCHMARK (10000 utilisateurs)

🎯 Pertinence Métier (Hit Ratio @ 5) :
 - SVD : 0.00 %
 - ALS : 14.39 %

⚡ Latence d'inférence (Temps de réponse / utilisateur) :
 - SVD : 2431.32 ms
 - ALS : 24.44 ms

💾 Empreinte Mémoire (RAM estimée pour le déploiement) :
 - SVD : 0.00 Mo
 - ALS : 69.41 Mo


In [7]:
print("\n💾 Empreinte Mémoire RÉELLE (RAM estimée pour le déploiement) :")

# 1. On utilise .nbytes pour SVD et ALS car ils ont réussi à s'entraîner
size_svd = model_svd.pu.nbytes + model_svd.qi.nbytes
size_als = model_als.user_factors.nbytes + model_als.item_factors.nbytes

# 2. Pour KNN, on calcule la taille de la matrice carrée (Utilisateurs x Utilisateurs x 8 octets)
n_users = trainset_surprise.n_users
size_knn_theorique = n_users * n_users * 8

print(f" - ALS : {size_als / (1024*1024):.2f} Mo (Gagnant !)")
print(f" - SVD : {size_svd / (1024*1024):.2f} Mo")
print(f" - KNN : {size_knn_theorique / (1024*1024*1024):.2f} Go  <-- (Taille théorique qui a causé le crash de la machine !)")



💾 Empreinte Mémoire RÉELLE (RAM estimée pour le déploiement) :
 - ALS : 69.41 Mo (Gagnant !)
 - SVD : 138.82 Mo
 - KNN : 776.82 Go  <-- (Taille théorique qui a causé le crash de la machine !)


In [8]:
import pickle
import numpy as np

print("💾 SAUVEGARDE DU MODÈLE GAGNANT (ALS) POUR LA PRODUCTION...")

# --- MODIFIE CE CHEMIN VERS LE DOSSIER DE TON DRIVE ---
SAVE_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/Collaborative_best"

# 1. Sauvegarde des matrices (Les "Cerveaux" du modèle) en format ultra-rapide .npy
np.save(f"{SAVE_DIR}als_user_factors.npy", model_als.user_factors)
np.save(f"{SAVE_DIR}als_item_factors.npy", model_als.item_factors)

# 2. Sauvegarde des Dictionnaires de traduction (Essentiel pour l'API)
# Pour retrouver à quel "vrai" user_id correspond la ligne 0, 1, 2...
with open(f"{SAVE_DIR}als_user_mapping.pkl", 'wb') as f:
    pickle.dump(users.cat.categories.values, f)
    
with open(f"{SAVE_DIR}als_item_mapping.pkl", 'wb') as f:
    pickle.dump(items.cat.categories.values, f)

print(f"✅ Modèle sauvegardé avec succès dans : {SAVE_DIR}")
print("Le notebook Filtrage Collaboratif est officiellement terminé !")

💾 SAUVEGARDE DU MODÈLE GAGNANT (ALS) POUR LA PRODUCTION...
✅ Modèle sauvegardé avec succès dans : /content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/Collaborative_best
Le notebook Filtrage Collaboratif est officiellement terminé !
